In [1]:
import os
import pandas as pd
from tqdm import tqdm

In [6]:
print(os.listdir("/kaggle/input/datasets"))

['mayarjao']


In [7]:
print(os.listdir("/kaggle/input/datasets/mayarjao"))

['arabic-tts']


In [8]:
base_path = "/kaggle/input/datasets/mayarjao/arabic-tts"

In [9]:
print(os.listdir(base_path))

['arabic_tts']


In [13]:
df = pd.read_csv(os.path.join(base_path, "metadata.csv"), sep="|", header=None)
print(df.head())

                                0                             1
0      هو يحبّها و هي تحبّه أيضا.  common_voice_ar_24203362.wav
1    من الممكن أنها لن تأتي غداً.  common_voice_ar_22931432.wav
2                       إبنك بطل.  common_voice_ar_26338992.wav
3       جاء ذلك الغبيّ يبحث عنّي.  common_voice_ar_35782277.wav
4  سامي يعلم أنّ ليلى غاضبة عليه.  common_voice_ar_24259106.wav


In [15]:
audio_dir = os.path.join(base_path, "wavs")

df["audio_path"] = df["file_name"].apply(
    lambda x: os.path.join(audio_dir, x + ".wav")
)

In [21]:
base_path = "/kaggle/input/datasets/mayarjao/arabic-tts/arabic_tts"
audio_dir = os.path.join(base_path, "wavs")

df = pd.read_csv(os.path.join(base_path, "metadata.csv"), sep="|", header=None)
df.columns = ["text", "file_name"]

df = df.dropna()
df["audio_path"] = df["file_name"].apply(lambda x: os.path.join(audio_dir, x))

print(df.head())
print("Total samples:", len(df))

                             text                     file_name  \
0      هو يحبّها و هي تحبّه أيضا.  common_voice_ar_24203362.wav   
1    من الممكن أنها لن تأتي غداً.  common_voice_ar_22931432.wav   
2                       إبنك بطل.  common_voice_ar_26338992.wav   
3       جاء ذلك الغبيّ يبحث عنّي.  common_voice_ar_35782277.wav   
4  سامي يعلم أنّ ليلى غاضبة عليه.  common_voice_ar_24259106.wav   

                                          audio_path  
0  /kaggle/input/datasets/mayarjao/arabic-tts/ara...  
1  /kaggle/input/datasets/mayarjao/arabic-tts/ara...  
2  /kaggle/input/datasets/mayarjao/arabic-tts/ara...  
3  /kaggle/input/datasets/mayarjao/arabic-tts/ara...  
4  /kaggle/input/datasets/mayarjao/arabic-tts/ara...  
Total samples: 78720


In [22]:
df["exists"] = df["audio_path"].apply(os.path.exists)
print(df["exists"].value_counts())

exists
True    78720
Name: count, dtype: int64


In [23]:
df = df[df["exists"]].drop(columns=["exists"])
print("Remaining samples:", len(df))

Remaining samples: 78720


In [24]:
import re

def clean_arabic_text(text):
    text = str(text).strip()

    # إزالة التشكيل
    text = re.sub(r'[ًٌٍَُِّْـ]', '', text)

    # توحيد الحروف
    text = re.sub(r'[إأآا]', 'ا', text)
    text = re.sub(r'ى', 'ي', text)
    text = re.sub(r'ؤ', 'و', text)
    text = re.sub(r'ئ', 'ي', text)

    # إزالة أي شيء ليس عربي أو مسافة
    text = re.sub(r'[^\u0600-\u06FF\s]', '', text)

    # إزالة المسافات الزائدة
    text = re.sub(r'\s+', ' ', text).strip()

    return text

df["text"] = df["text"].apply(clean_arabic_text)
df = df[df["text"].str.len() > 0]

print(df.head())
print("Remaining after text cleaning:", len(df))

                           text                     file_name  \
0       هو يحبها و هي تحبه ايضا  common_voice_ar_24203362.wav   
1    من الممكن انها لن تاتي غدا  common_voice_ar_22931432.wav   
2                      ابنك بطل  common_voice_ar_26338992.wav   
3        جاء ذلك الغبي يبحث عني  common_voice_ar_35782277.wav   
4  سامي يعلم ان ليلي غاضبة عليه  common_voice_ar_24259106.wav   

                                          audio_path  
0  /kaggle/input/datasets/mayarjao/arabic-tts/ara...  
1  /kaggle/input/datasets/mayarjao/arabic-tts/ara...  
2  /kaggle/input/datasets/mayarjao/arabic-tts/ara...  
3  /kaggle/input/datasets/mayarjao/arabic-tts/ara...  
4  /kaggle/input/datasets/mayarjao/arabic-tts/ara...  
Remaining after text cleaning: 78719


In [25]:
import soundfile as sf
from tqdm import tqdm

tqdm.pandas()

def get_duration(path):
    try:
        info = sf.info(path)
        return info.frames / info.samplerate
    except:
        return None

df["duration"] = df["audio_path"].progress_apply(get_duration)

print(df["duration"].describe())
print("Missing durations:", df["duration"].isna().sum())

100%|██████████| 78719/78719 [11:19<00:00, 115.81it/s]

count    78719.000000
mean         4.080597
std          1.469511
min          1.440000
25%          3.024036
50%          3.708027
75%          4.716009
max         22.068027
Name: duration, dtype: float64
Missing durations: 0


In [26]:
df = df[
    (df["duration"] >= 1.0) &
    (df["duration"] <= 12.0) &
    (df["text"].str.len() >= 2) &
    (df["text"].str.len() <= 120)
].copy()

print("Remaining after filtering:", len(df))
print(df[["text", "duration"]].head())

Remaining after filtering: 78717
                           text  duration
0       هو يحبها و هي تحبه ايضا  4.968027
1    من الممكن انها لن تاتي غدا  3.936009
2                      ابنك بطل  3.744036
3        جاء ذلك الغبي يبحث عني  4.788027
4  سامي يعلم ان ليلي غاضبة عليه  4.896009


In [27]:
def get_sample_rate(path):
    try:
        info = sf.info(path)
        return info.samplerate
    except:
        return None

sample_rates = df["audio_path"].head(200).apply(get_sample_rate)
print(sample_rates.value_counts())

audio_path
22050    200
Name: count, dtype: int64


In [28]:
output_audio_dir = "/kaggle/working/audio_16k"
os.makedirs(output_audio_dir, exist_ok=True)

In [29]:
import subprocess
from tqdm import tqdm

def resample_audio(input_path, output_path):
    cmd = [
        "ffmpeg",
        "-loglevel", "quiet",
        "-y",
        "-i", input_path,
        "-ac", "1",
        "-ar", "16000",
        "-sample_fmt", "s16",
        output_path
    ]
    subprocess.run(cmd)

In [30]:
sample_df = df.head(1000).copy()

new_paths = []

for path in tqdm(sample_df["audio_path"]):
    file_name = os.path.basename(path)
    out_path = os.path.join(output_audio_dir, file_name)

    resample_audio(path, out_path)
    new_paths.append(out_path)

sample_df["audio_path"] = new_paths

100%|██████████| 1000/1000 [02:23<00:00,  6.99it/s]


In [31]:
import os
import subprocess
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm.auto import tqdm
import pandas as pd

# =========================
# Config
# =========================
output_audio_dir = "/kaggle/working/audio_16k"
os.makedirs(output_audio_dir, exist_ok=True)

errors_csv_path = "/kaggle/working/resample_errors.csv"
success_csv_path = "/kaggle/working/resample_success.csv"

MAX_WORKERS = os.cpu_count() or 4
print(f"Using {MAX_WORKERS} workers")

# =========================
# Prepare jobs
# =========================
# df لازم يكون فيه audio_path و file_name
work_df = df.copy()

# المسار الجديد لكل ملف بعد التحويل
work_df["resampled_path"] = work_df["file_name"].apply(
    lambda x: os.path.join(output_audio_dir, x)
)

# no redundancy:
# اشتغل فقط على الملفات اللي output بتاعها مش موجود
to_process_df = work_df[~work_df["resampled_path"].apply(os.path.exists)].copy()
already_done_df = work_df[work_df["resampled_path"].apply(os.path.exists)].copy()

print(f"Already converted: {len(already_done_df)}")
print(f"To process now:    {len(to_process_df)}")
print(f"Total:             {len(work_df)}")

# =========================
# Worker function
# =========================
def resample_one(row):
    input_path, output_path = row

    # safety check
    if os.path.exists(output_path):
        return {
            "status": "skipped",
            "input_path": input_path,
            "output_path": output_path,
            "error": ""
        }

    cmd = [
        "ffmpeg",
        "-nostdin",
        "-loglevel", "error",
        "-y",
        "-i", input_path,
        "-ac", "1",
        "-ar", "16000",
        "-sample_fmt", "s16",
        output_path
    ]

    try:
        result = subprocess.run(
            cmd,
            stdout=subprocess.DEVNULL,
            stderr=subprocess.PIPE,
            text=True
        )

        if result.returncode == 0 and os.path.exists(output_path):
            return {
                "status": "success",
                "input_path": input_path,
                "output_path": output_path,
                "error": ""
            }
        else:
            return {
                "status": "failed",
                "input_path": input_path,
                "output_path": output_path,
                "error": result.stderr[-1000:] if result.stderr else "Unknown ffmpeg error"
            }

    except Exception as e:
        return {
            "status": "failed",
            "input_path": input_path,
            "output_path": output_path,
            "error": str(e)
        }

# =========================
# Run parallel processing
# =========================
jobs = list(zip(to_process_df["audio_path"].tolist(), to_process_df["resampled_path"].tolist()))

success_records = []
error_records = []

# لو فيه ملفات معمولة بالفعل، اعتبرها success من باب التتبع
if len(already_done_df) > 0:
    success_records.extend(
        {
            "status": "already_exists",
            "input_path": inp,
            "output_path": out,
            "error": ""
        }
        for inp, out in zip(already_done_df["audio_path"], already_done_df["resampled_path"])
    )

if len(jobs) > 0:
    with ProcessPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = [executor.submit(resample_one, job) for job in jobs]

        for i, future in enumerate(tqdm(as_completed(futures), total=len(futures), desc="Resampling"), 1):
            result = future.result()

            if result["status"] in ["success", "skipped"]:
                success_records.append(result)
            else:
                error_records.append(result)

            # checkpoint كل 1000 ملف
            if i % 1000 == 0:
                pd.DataFrame(success_records).to_csv(success_csv_path, index=False)
                pd.DataFrame(error_records).to_csv(errors_csv_path, index=False)

# save final logs
pd.DataFrame(success_records).to_csv(success_csv_path, index=False)
pd.DataFrame(error_records).to_csv(errors_csv_path, index=False)

print("\nDone.")
print(f"Success/Skipped/Existing: {len(success_records)}")
print(f"Failed:                   {len(error_records)}")

# =========================
# Build final dataframe
# =========================
# استخدم فقط الملفات التي تم تحويلها فعليًا أو كانت موجودة بالفعل
work_df["resampled_exists"] = work_df["resampled_path"].apply(os.path.exists)
final_df = work_df[work_df["resampled_exists"]].copy()

# حدّث audio_path إلى النسخة الجديدة 16k
final_df["audio_path"] = final_df["resampled_path"]

# نظّف الأعمدة الوسيطة
final_df = final_df.drop(columns=["resampled_exists"])

print(f"Final usable samples after resampling: {len(final_df)}")
print(final_df[["text", "file_name", "audio_path"]].head())

Using 4 workers
Already converted: 1000
To process now:    77717
Total:             78717


Resampling:   0%|          | 0/77717 [00:00<?, ?it/s]


Done.
Success/Skipped/Existing: 78717
Failed:                   0
Final usable samples after resampling: 78717
                           text                     file_name  \
0       هو يحبها و هي تحبه ايضا  common_voice_ar_24203362.wav   
1    من الممكن انها لن تاتي غدا  common_voice_ar_22931432.wav   
2                      ابنك بطل  common_voice_ar_26338992.wav   
3        جاء ذلك الغبي يبحث عني  common_voice_ar_35782277.wav   
4  سامي يعلم ان ليلي غاضبة عليه  common_voice_ar_24259106.wav   

                                          audio_path  
0  /kaggle/working/audio_16k/common_voice_ar_2420...  
1  /kaggle/working/audio_16k/common_voice_ar_2293...  
2  /kaggle/working/audio_16k/common_voice_ar_2633...  
3  /kaggle/working/audio_16k/common_voice_ar_3578...  
4  /kaggle/working/audio_16k/common_voice_ar_2425...  


In [32]:
import soundfile as sf

sample_check = final_df["audio_path"].sample(20, random_state=42).tolist()

for p in sample_check:
    info = sf.info(p)
    print(os.path.basename(p), info.samplerate, info.channels, info.subtype)

common_voice_ar_24159862.wav 16000 1 PCM_16
common_voice_ar_24056316.wav 16000 1 PCM_16
common_voice_ar_24155781.wav 16000 1 PCM_16
common_voice_ar_24054109.wav 16000 1 PCM_16
common_voice_ar_24024567.wav 16000 1 PCM_16
common_voice_ar_24101843.wav 16000 1 PCM_16
common_voice_ar_24164330.wav 16000 1 PCM_16
common_voice_ar_19471912.wav 16000 1 PCM_16
common_voice_ar_24076796.wav 16000 1 PCM_16
common_voice_ar_24166098.wav 16000 1 PCM_16
common_voice_ar_24165780.wav 16000 1 PCM_16
common_voice_ar_19465382.wav 16000 1 PCM_16
common_voice_ar_24058404.wav 16000 1 PCM_16
common_voice_ar_24043885.wav 16000 1 PCM_16
common_voice_ar_22251596.wav 16000 1 PCM_16
common_voice_ar_19210256.wav 16000 1 PCM_16
common_voice_ar_24052242.wav 16000 1 PCM_16
common_voice_ar_24082373.wav 16000 1 PCM_16
common_voice_ar_24045889.wav 16000 1 PCM_16
common_voice_ar_32700065.wav 16000 1 PCM_16


In [33]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(final_df, test_size=0.2, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

print("Train:", len(train_df))
print("Val:  ", len(val_df))
print("Test: ", len(test_df))

train_df.to_csv("/kaggle/working/train.csv", index=False)
val_df.to_csv("/kaggle/working/val.csv", index=False)
test_df.to_csv("/kaggle/working/test.csv", index=False)

Train: 62973
Val:   7872
Test:  7872


In [34]:
final_df["text_len"] = final_df["text"].str.len()

print(final_df[["text_len", "duration"]].describe())

           text_len      duration
count  78717.000000  78717.000000
mean      27.201773      4.080383
std       15.009720      1.468124
min        2.000000      1.440000
25%       17.000000      3.024036
50%       23.000000      3.708027
75%       34.000000      4.716009
max      113.000000     10.871973


In [35]:
import re

def arabic_ratio(text):
    arabic_chars = re.findall(r'[\u0600-\u06FF]', text)
    return len(arabic_chars) / max(1, len(text))

final_df["arabic_ratio"] = final_df["text"].apply(arabic_ratio)

print(final_df["arabic_ratio"].describe())

count    78717.000000
mean         0.842982
std          0.037962
min          0.709677
25%          0.818182
50%          0.840000
75%          0.863636
max          1.000000
Name: arabic_ratio, dtype: float64


In [36]:
import os

print(os.listdir("/kaggle/working"))

['audio_16k', 'resample_errors.csv', '.virtual_documents', 'test.csv', 'train.csv', 'val.csv', 'resample_success.csv']


In [37]:
import os

for root, dirs, files in os.walk("/kaggle/working"):
    print(root, len(files))
    print(files[:10])
    print("-" * 60)

/kaggle/working 5
['resample_errors.csv', 'test.csv', 'train.csv', 'val.csv', 'resample_success.csv']
------------------------------------------------------------
/kaggle/working/audio_16k 78717
['common_voice_ar_24024875.wav', 'common_voice_ar_24037157.wav', 'common_voice_ar_20843873.wav', 'common_voice_ar_24059441.wav', 'common_voice_ar_24158703.wav', 'common_voice_ar_24157406.wav', 'common_voice_ar_24092026.wav', 'common_voice_ar_24065490.wav', 'common_voice_ar_24075236.wav', 'common_voice_ar_24065742.wav']
------------------------------------------------------------
/kaggle/working/.virtual_documents 1
['__notebook_source__.ipynb']
------------------------------------------------------------
